### Classification using sklearn

- This week focuses on sklearn functionality for implementing classification algorithms.

### Topics Covered

- Least Squares Classification
- Perceptron
- Logistic Regression Classifier
  - regularization
  - multiclass setting
  - multilabel setting
  - multi-output setting

### Classification Metrics

- Different metrics used to evaluate classification models.

### Cross Validation

- Cross validation and hyperparameter search work similarly to regression.
- Some CV strategies are specifically designed for classification tasks.

### Types of Classification APIs in sklearn

sklearn classification APIs are broadly divided into two categories based on how they work internally.

### Generic APIs

- SGD Classifier
  - Uses Gradient Descent for optimization.
  - Works by updating model parameters step-by-step to reduce error.
  - Flexible because we can choose different loss functions.
  - Need to manually specify the loss function depending on the task.
  
- Example:
  - `loss="log_loss"` → behaves like Logistic Regression
  - `loss="perceptron"` → behaves like Perceptron

### Specific APIs

These algorithms have their own dedicated implementations and specialized optimization methods.

- Logistic Regression
  - Used for classification using probability estimation.
  - Has built-in optimization solvers.

- Perceptron
  - Simple linear classifier.
  - Mainly used for binary classification.

- Ridge Classifier (for Least Squares Classification)
  - Uses ridge regularization to reduce overfitting.
  - Faster for some linear classification tasks.

- K-Nearest Neighbours (KNN)
  - Predicts output using nearest data points.
  - No training phase in the traditional sense.

- Support Vector Machines (SVMs)
  - Finds the best boundary separating classes.
  - Effective for high-dimensional data.

- Naive Bayes
  - Probabilistic classifier based on Bayes theorem.
  - Assumes features are independent.

### Main Difference

- Generic API:
  - More flexible.
  - Requires selecting loss function manually.

- Specific API:
  - Easier to use for a particular algorithm.
  - Uses specialized solvers optimized for that algorithm.

### Scikit-Learn Model Workflow: Train, Predict & Evaluate

Ye Scikit-Learn library ka standard machine learning pipeline hai jo lagbhag har classification model (Logistic Regression, SVM, KNN, etc.) me follow hota hai.

#### 1. Model Training: `fit(X, y)`
* **Ye kya hai?** Ye model ka "learning phase" ya training phase hai. Is function ko call karne par, model aapke features (`X`) aur labels (`y`) ko analyze karke unke beech ka relationship (pattern) samajhta hai.
* **Kyu aur Kab use karte hai?** Data preprocessing hone ke turant baad, model ko sikhane ke liye. Jab tak `fit()` run nahi hoga, model "untrained" rahega aur kuch predict nahi kar payega.
* **Iske piche ka Maths:** 
  * `X` (Features): Ek 2D matrix hota hai. Dimensions: `(n_samples, n_features)`.
  * `y` (Target/Labels): Ek 1D array hota hai jisme original answers hote hain. Dimensions: `(n_samples,)`.
  * **Background Maths:** Algorithm is data ka use karke ek Cost Function / Loss Function ko minimize karta hai. Mathematical terms me, model parameters (weights $W$ aur bias $b$) ki optimal values find karta hai jo error ko sabse kam karein.
* **Note:** Parameters jaise `coef_init`, `intercept_init` advance use cases ke liye hote hain jab aap algorithm ko kisi specific starting point se initialize karna chahte ho.

#### 2. Prediction Phase
Model train hone ke baad, hum usko naye/unseen data par test karte hain. Iske liye do main functions hote hain:

**A. `predict(X)`**
* **Ye kya hai?** Ye function seedha "Class Label" (jaise 0 ya 1, Spam ya Not Spam) predict karke deta hai.
* **Kyu aur Kab use karte hai?** Jab aapko end-user ko final result dikhana ho.
* **Background Maths:** Classification models me, ye generally `decision_function` ya probabilities par ek threshold apply karke output deta hai (e.g., agar probability > 0.5 hai, toh class 1).

**B. `decision_function(X)`**
* **Ye kya hai?** Ye final class label dene ke bajaye ek "Confidence Score" (raw value) return karta hai. Ye score batata hai ki ek data point classification ki boundary (hyperplane) se kitni door hai. 
* **Kyu aur Kab use karte hai?** Jab aapko model ka "Threshold" apne hisaab se change karna ho (jaise medical data me aap zyada strict hona chahte ho) ya phir aapko ROC (Receiver Operating Characteristic) curve plot karna ho.
* **Iske piche ka Maths:** 
  Linear classifiers ke case me raw score is formula se nikalta hai: 
  $$ f(X) = W^T X + b $$
  Jahan $W$ aapke model weights hain aur $b$ intercept hai. Jitni badi value (positive ya negative), model utna hi confident hota hai.

#### 3. Evaluation: `score(X, y)`
* **Ye kya hai?** Ye function internally pehle `X` par `predict(X)` run karta hai, aur fir uski tulna actual `y` se karta hai. Classification models me ye by default **Mean Accuracy** return karta hai.
* **Kyu aur Kab use karte hai?** Training ke baad test data (unseen data) par model ki performance check karne ke liye.
* **Iske piche ka Maths:** 
  Classification accuracy ka formula bahut simple hai:
  $$ \text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}} $$
  Mathematical terms me:
  $$ \text{Accuracy} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}(\hat{y}_i = y_i) $$
  (Jahan $\mathbb{1}$ ek indicator function hai jo true hone par 1 deta hai, aur false hone par 0).

#### There a few common miscellaneous methods as follows:


* `get_params ([deep])` : **gets parameter for this estimator.**

* `set_params ( ** params)` : **sets the parameters of this estimator.**

* `densify()` : **converts coefficient matrix to dense array format.**

* `sparsify()` : **converts coefficient matrix to sparse format.**

### Least Square Classification (LSC) via RidgeClassifier

`RidgeClassifier` koi completely naya classification algorithm nahi hai, balki ye famous **Ridge Regression** ka hi ek variant hai. Isme hum continuous values predict karne wali math ko modify karke classes (categories) predict karte hain.

#### 1. Binary Classification (Do Classes ke liye)
* **Ye kya hai aur kaise kaam karta hai?** 
  Agar aapke target labels `0` aur `1` hain, toh classifier sabse pehle inko mathematically `-1` aur `1` mein badal deta hai. Iske baad, model is problem ko ek normal regression task maan leta hai aur aisi weights ($w$) find karta hai jo data ke hisaab se best fit hon.
* **Iske piche ka Maths (Objective Function):**
  Kyunki ye based on Ridge Regressor hai, iska main objective "Penalized Residual Sum of Squares (RSS)" ko minimize karna hota hai. 
  Iska equation ye hai:
  $$ \min_{w} ||Xw - y||_2^2 + \alpha ||w||_2^2 $$
  * **$X$**: Aapka input data (Features).
  * **$w$**: Model ke weights ya coefficients.
  * **$y$**: Target labels (jo ab `-1` aur `1` hain).
  * **$||Xw - y||_2^2$**: Ye actual error hai (Actual value aur predicted value ke beech ka squared difference). Ise model kam karna chahta hai.
  * **$\alpha ||w||_2^2$**: Ye **L2 Regularization** term hai. Sklearn mein $\alpha$ (`alpha`) ko regularization rate/strength kaha jata hai. Ye weights ($w$) ko bahut bada hone se rokti hai, jisse model **Overfit** nahi hota.
* **Prediction Phase:** 
  Regression output humesha ek continuous number deta hai (jaise 0.8, -1.5, 2.3). Toh class kaise pata chalegi? 
  Bahut simple: Model sirf output ka **Sign (+ ya -)** check karta hai.
  * Agar output $> 0$ (Positive) $\rightarrow$ Model predict karega Class `1`
  * Agar output $< 0$ (Negative) $\rightarrow$ Model predict karega Class `-1`

#### 2. Multiclass Classification (Do se zyada Classes ke liye)
* **Ye kya hai?**
  Jab targets 3 ya usse zyada hon (e.g., Red, Green, Blue), toh sklearn isko "Multi-output Regression" ki tarah treat karta hai. Har class ke liye ek alag score calculate hota hai.
* **Prediction Phase:**
  Jis bhi class ka regression output (score) sabse **highest** (maximum) hota hai, model usko final prediction maan leta hai. Is process ko mathematically `argmax` kehte hain.

#### 3. Important Points & Solvers
* **Kyun aur Kab use karein?** Ye linearly separable data ke liye kafi fast aur effective hai. L2 regularization ki wajah se ye multicollinearity (jab features aapas me highly correlated hon) ko achhe se handle kar leta hai.
* **Solvers:** Optimization (minimize) karne ke liye sklearn ke paas alag-alag mathematical solvers hote hain (jaise `cholesky`, `lsqr`, `sag`). Agar aapka dataset bahut bada hai, toh `sag` (Stochastic Average Gradient) use karna model ko fast banata hai.
* **Edge Case Note:** Kyunki ye basically ek regression algorithm hai, `RidgeClassifier` probabilities output nahi karta (isliye isme `.predict_proba()` method kaam nahi karta). Ye seedha classes batata hai.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

x , y = load_iris(as_frame=True , return_X_y=True)
x_train , x_test , y_train , y_test = train_test_split(x,y , test_size=0.2 , random_state=42)

In [24]:
from sklearn.linear_model import RidgeClassifier

ridge_classifier = RidgeClassifier(alpha=0.1)
ridge_classifier.fit(x_train , y_train)

,"alpha alpha: float, default=1.0Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If set to false, nointercept will be used in calculations (e.g. data is expected to bealready centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.The default value is determined by scipy.sparse.linalg.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard scipy.linalg.solve function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in scipy.sparse.linalg.cg. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine scipy.sparse.linalg.lsqr. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its unbiased and more flexible version named SAGA. Both methods use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from sklearn.preprocessing. .. versionadded:: 0.17 Stochastic Average Gradient descent solver. .. versionadded:: 0.19 SAGA solver.- 'lbfgs' uses L-BFGS-B algorithm implemented in `scipy.optimize.minimize`. It can be used only when `positive` is True.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details.",None


In [25]:
preds = ridge_classifier.predict(x_test)
print("Model Prediction on Test Dataset",preds)
print("Model Score on test Dataset",ridge_classifier.score(x_test , y_test))

Model Prediction on Test Dataset [1 0 2 2 1 0 2 2 1 1 2 0 0 0 0 2 2 1 1 2 0 2 0 2 2 2 1 2 0 0]
Model Score on test Dataset 0.8666666666666667


### Optimization Solvers in RidgeClassifier: A Deep Mathematical Dive

Ridge Regression ki Normal Equation hoti hai: 
$$ w = (X^T X + \alpha I)^{-1} X^T y $$
In solvers ka main kaam is equation ko solve karke optimal weights ($w$) nikalna hai. Aaiye inke piche ki maths ko deeply samajhte hain:

#### 1. `svd` (Singular Value Decomposition)
* **Ye kya hai?** Ye sabse stable aur accurate math trick hai. Ye feature matrix $X$ ko teen alag matrices me tod deta hai.
* **Iske piche ka Maths:** 
  Linear Algebra ke hisaab se, kisi bhi matrix $X$ ko aise likha ja sakta hai:
  $$ X = U \Sigma V^T $$
  (Jahan $U$ aur $V$ orthogonal matrices hain, aur $\Sigma$ diagonal matrix hai jisme singular values hoti hain).
  Jab hum isko Ridge ki Normal Equation me daalte hain, toh $X^T X$ calculate karne ki zaroorat hi nahi padti. Equation directly ban jati hai:
  $$ w = V (\Sigma^2 + \alpha I)^{-1} \Sigma U^T y $$
* **Fayda kya hai?** Kyunki $\Sigma$ ek diagonal (straight line) matrix hai, iska inverse nikalna (matlab $\frac{1}{\text{value}}$ karna) bahut aasan hai. Ye solver tab use hota hai jab data bahut complex ho aur baki solvers fail (unstable) ho rahe hon.

#### 2. `cholesky` (The Default Closed-Form Solver)
* **Ye kya hai?** Ye `scipy.linalg.solve` ka use karta hai aur Matrix Decomposition ki ek aur technique lagata hai jise **Cholesky Decomposition** kehte hain. 
* **Iske piche ka Maths:** 
  Normal equation $w = (X^T X + \alpha I)^{-1} X^T y$ ko solve karne ke liye hume $Aw = b$ ke form me ek linear system solve karna hota hai, jahan $A = (X^T X + \alpha I)$ aur $b = X^T y$.
  Kyunki matrix $A$ symmetric (dono taraf se same) aur positive-definite (regularization $\alpha$ ki wajah se) hoti hai, Cholesky decomposition is matrix $A$ ko aise tod deta hai:
  $$ A = L L^T $$
  (Jahan $L$ ek Lower Triangular Matrix hai). 
  Triangular matrices me equations solve karna (Back-substitution) computer ke liye fraction of seconds ka kaam hai.
* **Kab use karein?** Ye standard aur fast hai medium-sized datasets ke liye jahan $X^T X$ memory me fit aa sake.

#### 3. `sparse_cg` (Conjugate Gradient)
* **Ye kya hai?** Jab data bahut bada hota hai (e.g., text data, NLP), toh usme zyada values `0` hoti hain (Sparse Data). Wahan matrix ka inverse nikalna memory crash kar dega. Ye ek **Iterative (step-by-step)** solver hai.
* **Iske piche ka Maths:** 
  Gradient Descent me model direct dhalaan (steepest descent) par neeche aata hai, jisme zig-zag hota hai. Lekin Conjugate Gradient (CG) me model **Orthogonal (Conjugate) directions** me move karta hai. 
  Iska matlab hai ki agar usne ek direction me error solve kar di, toh agle step me wo us purani direction ko kharab nahi karega. Ye $Aw = b$ ko iteratively solve karta hai bina matrix $A$ ko puri tarah memory me load kiye.

#### 4. `lsqr` (Least-Squares Routine - The Fastest for Sparse)
* **Ye kya hai?** Ye bhi ek iterative solver hai, bilkul `sparse_cg` jaisa, lekin mathematically thoda zyada advanced aur fast hai.
* **Iske piche ka Maths:** 
  Ye Golub-Kahan bidiagonalization process use karta hai. Normal equation $X^T X w = X^T y$ ko solve karne ke bajaye, ye internally ek equivalent par zyada stable system banata hai. Agar aapka matrix $X$ poorly conditioned hai (matlab columns ke beech aapas me jhol/correlation hai), toh `sparse_cg` galti kar sakta hai, par `lsqr` usko handle kar lega aur sabse tezi se converge karega.

#### 5. `sag` & `saga` (Stochastic Average Gradient)
* **Ye kya hai?** Stochastic Gradient Descent (SGD) ka naam suna hoga? Ye usika advanced "Average" version hai. Jab millions me data (rows) ho, tab ye lagate hain.
* **Iske piche ka Maths:** 
  Normal SGD har row (sample) ko dekh kar weights update karta hai:
  $$ w^{(t+1)} = w^{(t)} - \eta \nabla f_i(w) $$
  (Jahan $\nabla f_i$ ek row ka gradient hai). 
  Lekin `sag` (Stochastic Average Gradient) ek **Memory** maintain karta hai. Ye purane gradients ko yaad rakhta hai aur unka "Average" le kar weight update karta hai. Is wajah se iska rasta normal SGD se kam zig-zag hota hai aur ye bahut fast minimum par pahunchta hai.
* **`saga` kyu?** `saga` unbiased hai aur ye L1 regularization (Lasso) ko bhi support karta hai. Jab dataset bahut massive ho, tab 'saga' sabse best choice hoti hai.

#### 6. `lbfgs` (L-BFGS-B Algorithm)
* **Ye kya hai?** Ye ek **Quasi-Newton Method** hai. Agar Gradient Descent 1st derivative (Slope) use karta hai, toh ye mathematically 2nd derivative (Curvature) ka use karke direct target par koodne ki koshish karta hai.
* **Iske piche ka Maths:** 
  Exact 2nd derivative (Hessian Matrix - $\nabla^2 f$) nikalna almost namumkin aur memory intensive hota hai badi matrices ke liye. L-BFGS **(Limited-memory Broyden–Fletcher–Goldfarb–Shanno)** Hessian matrix ko compute nahi karta, balki pichle kuch steps (Limited memory) ke gradients ka use karke Hessian ko **approximate (guess)** kar leta hai. 
* **Kab use karein?** Isme ek extra feature 'B' (Bound) laga hai. Iska matlab agar aapko model ko strictly force karna hai ki "Weights sirf Positive (+) hi hone chahiye" (forced positive coefficients), tabhi sklearn me RidgeClassifier isko use karne ki anumati deta hai.

#### How to make RidgeClassifier select the solver automatically?
```python
ridge_classifier = RidgeClassifier(auto=True)
```
**Chooses solver automatically based on data type**

### Intercept Estimation in RidgeClassifier

`fit_intercept` parameter ye tay karta hai ki kya model ko intercept ($b$) ya bias term calculate karna chahiye ya nahi. Scikit-Learn mein iski default value `True` hoti hai.

#### 1. Ye kya hai? (The Intuition)
* **Intercept ($b$) ka kaam:** Koi bhi linear model ek line (ya hyperplane) banata hai. Line ki equation hoti hai: 
  $$ y = WX + b $$
  Yahan $b$ intercept hai. Ye model ko flexibility deta hai ki line origin $(0,0)$ se pass hone ke bajaye, y-axis par kahin se bhi shuru ho sake.
* Agar $b=0$ hoga (matlab intercept nahi hai), toh line ko zabardasti origin $(0,0)$ se hi guzarna padega.

#### 2. Kya Intercept Estimate karna zaroori hai?
* **Short Answer:** Haan, zyada tar cases mein.
* **Kyun?** Agar aapke data ka center origin $(0,0)$ par nahi hai, aur aap intercept nahi dete, toh model ek ghatiya line fit karega (wo zabardasti use $(0,0)$ se khinchne ki koshish karega), jisse error badh jayegi aur classification boundary galat banegi.

#### 3. `fit_intercept=False` kab use karein? (The Mathematical Exception)
Lecture ka statement: *"If data is already centered, set `fit_intercept` as false..."*
* **Centered Data kya hota hai?** Agar aapne data processing ke time par har feature mein se uska Mean (average) minus kar diya hai:
  $$ X_{\text{centered}} = X - \text{Mean}(X) $$
  Toh data ka average 0 ban jata hai (matlab pura data origin ke aas-paas shift ho gaya).
* **Fayda:** Agar data origin ke hi chaaron taraf hai, toh best fit line waise hi $(0,0)$ se paas hogi. Is case mein $b=0$ by default ho jata hai.
* Tab aap `fit_intercept=False` kar sakte ho. Iska fayda ye hai ki model ko ek variable ($b$) kam calculate karna padta hai, toh calculation thodi fast ho jati hai aur memory bachti hai.

#### 4. The Default Behaviour
```python
ridge_classifier = RidgeClassifier(fit_intercept=True)

### Perceptron Classification: The First Neural Network

Perceptron ek bahut hi simple linear binary classifier hai. Ye ek straight line (hyperplane) khinch kar data ko do hisson me baatne ki koshish karta hai. 

#### 1. Ye kya hai aur iske piche ka Maths?
* **Prediction Rule:** Model data $X$ aur weights $W$ ka dot product nikalta hai. 
  $$ z = W^T X + b $$
  Fir is par ek **Step Function** lagata hai:
  * Agar $z > 0$, toh output (Class) $= 1$
  * Agar $z \le 0$, toh output (Class) $= -1$

* **Learning Rule (Weight Update):** Model ek-ek karke data point dekhta hai. 
  * Agar prediction **sahi** hai $\rightarrow$ Weights me koi change nahi.
  * Agar prediction **galat** hai $\rightarrow$ Model apne weights ko is formula se update karta hai:
    $$ W_{\text{new}} = W_{\text{old}} + \eta \cdot y_i \cdot X_i $$
    (Jahan $\eta$ (eta) learning rate hai, $y_i$ actual class hai, aur $X_i$ input features hain). Ye update tab tak chalta hai jab tak line ekdam sahi jagah set na ho jaye.

#### 2. Perceptron aur SGDClassifier ka Connection (The Big Reveal)
Aapke lecture me ek code diya hai:
```python
Perceptron() == SGDClassifier(loss="perceptron", eta0=1, learning_rate="constant", penalty=None)

In [ ]:
from sklearn.linear_model import Perceptron , SGDRegressor

perceptron_classifier = Perceptron()

perceptron_classifier.fit(x_train,y_train)

preds = perceptron_classifier.predict(x_train)
print("predction on train dataset:",preds)

score = perceptron_classifier.score(x_test,y_test)
print("score on test dataset",score)



predction on train dataset: [0 0 0 0 0 2 0 0 0 0 2 0 0 0 0 1 2 2 0 2 0 2 0 0 2 0 0 0 0 1 2 0 0 0 0 0 0
 2 0 0 2 0 2 2 1 0 2 1 0 0 2 0 0 1 0 0 2 0 0 2 0 2 2 2 2 0 0 0 2 2 0 0 0 1
 2 0 2 2 0 0 0 2 0 2 0 2 0 2 0 0 0 0 0 1 0 0 2 2 0 0 2 1 0 2 0 0 2 2 0 2 1
 0 2 2 0 0 2 0 0 2]
score on test dataset 0.8


#### Using Perceptron with SGD training Style

In [48]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(
    loss="perceptron",
    penalty=None,
    learning_rate="constant",
    eta0=1
)
sgd_clf.fit(x_train,y_train)

sgd_clf.score(x_test,y_test)

0.9666666666666667

### Perceptron Customization Parameters (Deep Dive)

Bhale hi Perceptron basic algorithm hai, par `SGDClassifier` ke wrapper ke through hum isme modern Deep Learning techniques laga sakte hain. Inhe 3 groups me samajhte hain:

#### Group 1: The Regularization (Overfitting Rokne wale)
Jab model training data ka "ratta" maarne lagta hai, toh hum penalty lagate hain.

* **`penalty` (default = 'l2'):** 
  * Ye batata hai ki penalty kis mathematical formula se lagegi. 
  * `'l2'` (Ridge): Weights ko chota karta hai. Math: $\sum w^2$
  * `'l1'` (Lasso): Ye unwanted features ke weights ko exact $0$ kar deta hai (Feature selection). Math: $\sum |w|$
  * `'elasticnet'`: Ye dono L1 aur L2 ka mix hota hai.
* **`alpha` (default = 0.0001):** 
  * Ye penalty ka "Strictness Level" hai (humne Ridge me padha tha). 
  * Badi value = Zyada penalty. Chhoti value = Kam penalty. Model update ke waqt gradient me ye factor multiply hota hai.
* **`l1_ratio` (default = 0.15):** 
  * Ye sirf tab kaam karta hai jab `penalty='elasticnet'` ho. 
  * ElasticNet dono ka mix hai: `Total Penalty = (l1_ratio * L1) + ((1 - l1_ratio) * L2)`. 
  * Agar ratio 0.15 hai, matlab 15% power L1 (Lasso) ki hai, aur 85% power L2 (Ridge) ki hai.

#### Group 2: The Learning Engine (Seekhne ki Speed)

* **`eta0` (default = 1):** 
  * Ise **Learning Rate ($\eta$)** kehte hain. Ye batata hai ki jab model galti karega, toh wo apne weights me kitna bada badlaav (step) karega.
  * **Maths:** $W_{\text{new}} = W_{\text{old}} + \eta \cdot (\text{Error Gradient})$
  * Agar $\eta$ bahut bada hai, toh model step miss kar jayega (Zig-Zag). Agar bahut chota hai, toh training me sadiyan lag jayengi. Perceptron me traditionally ye 1 rakha jata hai.

#### Group 3: The Convergence & Stopping Rules (Kab rukna hai?)
Real-world data me classes perfect straight line se alag nahi hoti. Agar hum model ko nahi rokenge, toh wo infinite loop me fasa rahega (hamesha thodi error bachegi). Isliye ye rules hote hain:

* **`max_iter` (default = 1000):** 
  * Ye Maximum Epochs hain. Model pure dataset (saari rows) ko kitni baar dekhega. 1000 matlab pura data 1000 baar padha jayega. Model isse zyada aage nahi badhega, chahe error 0 na hui ho.
* **`tol` (default = 1e-3):** 
  * Tolerance level ($0.001$). Model har step ke baad apna naya Loss check karta hai. 
  * **Maths:** Agar $(Loss_{\text{old}} - Loss_{\text{new}}) < tol$ hai, matlab ab model ki accuracy lagbhag badhna band ho gayi hai (fayda nahi ho raha).
* **`n_iter_no_change` (default = 5):** 
  * Ye `tol` ke saath milkar kaam karta hai. Agar lagataar 5 baar (5 epochs tak) model ka improvement `tol` (0.001) se kam raha, toh model samajh jata hai ki ab aur seekhne ka koi scope nahi hai, aur wo training ko wahin rok (stop) deta hai.

#### Group 4: Early Stopping (Smart Training)
Ye Deep Learning ka sabse powerful concept hai!

* **`early_stopping` (default = False):** 
  * Agar True kiya, toh model chori-chipe training data ka kuch hissa "Test" lene ke liye bacha leta hai. Wo training sirf bache hue data par karta hai, aur apne aap ko us chhupe hue data par test karta rehta hai. Jab use lagta hai ki chhupe hue data par accuracy girne lagi (Overfitting shuru), wo training rok deta hai.
* **`validation_fraction` (default = 0.1):** 
  * Agar `early_stopping=True` hai, toh ye batata hai ki kitna data chhupana hai. $0.1$ ka matlab hai Total Training Data ka **10%** hissa Validation (Test) ke liye alag nikal liya jayega.
* **`fit_intercept` (default = True):** 
  * Ye hum pehle hi discuss kar chuke hain. Bias/Intercept ($b$) calculate karna hai ya nahi.

#### Summary of the Flow:
Model seekhna shuru karega (`eta0` ki speed se). Wo apne weights par penalty (`penalty`, `alpha`) lagata rahega taaki ratta na maare. Wo maximum 1000 baar data dekhega (`max_iter`), lekin agar lagataar 5 baar (`n_iter_no_change`) uski performance me 0.001 (`tol`) se zyada improvement nahi aaya, toh wo us se pehle hi training band kar dega!

### Advanced Training in Perceptron: `partial_fit` & `warm_start`

Jab aapke paas 100 GB ka data ho aur RAM sirf 8 GB ho, tab standard `.fit()` function Out-of-Memory (OOM) error dekar crash ho jayega. Yahan ye do techniques kaam aati hain.

---

#### 1. Iterative Training using `partial_fit()` (Online Learning)
* **Ye kya hai?** 
  `partial_fit` model ko thoda-thoda karke sikhata hai. Aap apne 100 GB data ko 1-1 GB ke 100 tukdon (batches) me baant lete ho. Model pehla batch dekhta hai, weights update karta hai, fir us batch ko memory se hata deta hai aur agla batch bulata hai.
* **Iske piche ka Maths:** 
  Normal Perceptron me update rule hota hai: $W_{t+1} = W_t + \eta y X$. 
  `partial_fit` is update ko sirf current "Batch" ke data par lagata hai. Isliye ye pichle batches ka knowledge $W_t$ me save rakhta hai.
* **Special Requirement:** 
  Jab aap pehli baar `partial_fit` call karte ho, toh model ko pata nahi hota ki aage chal kar aur kaun-kaun si classes (categories) aane wali hain. Isliye pehli baar mein `classes` parameter pass karna compulsory hota hai.

---

In [49]:
import numpy as np
from sklearn.linear_model import Perceptron

# Maan lo ye humara 4 rows ka total data hai
X_total = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
y_total = np.array([0, 0, 1, 1])

# Model initialize kiya
clf_partial = Perceptron()

# Hamein saari classes (categories) pehle se batani padengi
all_classes = np.array([0, 1])

# BATCH 1: Pehli 2 rows bheji
print("--- Training Batch 1 ---")
clf_partial.partial_fit(X_total[:2], y_total[:2], classes=all_classes)
print("Weights after Batch 1:", clf_partial.coef_)
print("Bias after Batch 1:", clf_partial.intercept_)

# BATCH 2: Agli 2 rows bheji (Notice: Yahan 'classes' pass karna zaroori nahi)
print("\n--- Training Batch 2 ---")
clf_partial.partial_fit(X_total[2:], y_total[2:])
print("Weights after Batch 2:", clf_partial.coef_)
print("Bias after Batch 2:", clf_partial.intercept_)

--- Training Batch 1 ---
Weights after Batch 1: [[-3. -4.]]
Bias after Batch 1: [-1.]

--- Training Batch 2 ---
Weights after Batch 2: [[4. 4.]]
Bias after Batch 2: [0.]


### Logistic Regression API in Scikit-Learn

Naam me 'Regression' hone ke bawajood ye ek **Classification Algorithm** hai. Ye output me seedha 0 ya 1 dene ke bajaye, **Probability (Sambhavna)** return karta hai (jaise 0.85 ya 85% chance ki ye Class 1 hai). Phir ek threshold (jaise 0.5) laga kar hum ise class me badal dete hain.

#### 1. Iske Alag-Alag Naam (Aur kyu?)
Lecture me 3 naam diye hain, inka meaning samajhna zaroori hai:
* **Logit Regression:** Kyunki ye 'Logit' function (Sigmoid function) ka use karke linear equation $W^T X + b$ ki value ko 0 aur 1 ke beech me dabata (squash karta) hai.
* **Maximum Entropy Classifier (MaxEnt):** Information Theory ke hisaab se, ye model us probability distribution ko chunta hai jiski "Entropy" (randomness) sabse zyada ho, bina purane rules tode. Matlab, jab tak data pakka proof na de, model khud se koi bias nahi banata (impartial rehta hai).
* **Log-Linear Classifier:** Kyunki iski Decision Boundary (jo line classes ko alag karti hai) ekdam straight/linear hoti hai, par iska output log-probabilities me aata hai.

#### 2. The Core Math: The Objective Function
Scikit-Learn ka Logistic Regression is function ko minimize (sabse kam) karne ki koshish karta hai:
$$ \arg \min_{w,c} \Big[ \text{Regularization Penalty} + C \times \text{Cross Entropy Loss} \Big] $$

Is equation ke 4 tukde hain, aaiye inme ghuste hain:
* **$\arg \min_{w,c}$**: Model ka aim hai un Best Weights ($w$) aur Intercept/Bias ($c$) ko dhoondhna jo is pure bracket ki value ko minimum kar dein.
* **Cross Entropy Loss (Log Loss):** Ye regression wale MSE (Mean Squared Error) ki jagah use hota hai. Agar aapka target $y$ (0 ya 1) hai aur model ki predicted probability $\hat{y}$ (0 se 1 ke beech) hai, toh loss ka formula hota hai:
  $$ \text{Loss} = - \sum \Big[ y \log(\hat{y}) + (1-y) \log(1-\hat{y}) \Big] $$
  *Intuition:* Agar actual class 1 hai, aur model ne probability 0.99 di, toh loss bahut kam aayega. Par agar usne 0.01 probability di (yani wo full confidence ke sath galat hai), toh loss infinity ki taraf shoot kar jayega. Model ko galat over-confidence par bhari fine lagta hai.
* **The Magic of Parameter $C$ (Very Important Gotcha):**
  Aapne RidgeClassifier me padha tha ki hum penalty ko badhane ke liye $\alpha$ (Alpha) badhate the. 
  Logistic Regression me sklearn $\alpha$ ki jagah **$C$** use karta hai. Aur yahan rule ulta hai! 
  $$ C = \frac{1}{\alpha} $$
  * Equation dhyan se dekho: $C$ loss ke sath multiply ho raha hai, penalty ke sath nahi.
  * **Agar $C$ bahut chota hai (e.g., 0.01):** Toh Loss term ki value gir jayegi, model Penalty par zyada focus karega $\rightarrow$ **High Regularization (Simple model, underfitting risk)**.
  * **Agar $C$ bahut bada hai (e.g., 1000):** Toh model Penalty ko ignore kar dega aur sirf Loss ko zero karne me lag jayega $\rightarrow$ **Low Regularization (Complex model, overfitting risk)**.

#### 3. Classification Strategies (Ye kaam kaise karta hai)
Ye API 3 tarah ke classification ko handle kar sakti hai:

* **A. Binary Classification:** Sirf 2 classes (0 ya 1). Piche direct **Sigmoid Function** use hota hai.
  $$ P(y=1|X) = \frac{1}{1 + e^{-(w^T X + c)}} $$

* **B. One-Vs-Rest (OVR):** Jab 3 ya usse zyada classes hon (e.g., Apple, Banana, Orange). 
  Ye piche 3 alag-alag binary classifiers banayega:
  1. Apple vs (Banana + Orange)
  2. Banana vs (Apple + Orange)
  3. Orange vs (Apple + Banana)
  Jiska score sabse highest aayega, model wo class output de dega. Ye thoda purana aur slow tareeqa hai.

* **C. Multinomial Logistic Regression (Softmax):** 
  Ye modern tareeqa hai. Isme OVR ki tarah alag-alag model nahi bante, balki ek hi model ek saath saari classes ki probabilities nikalta hai jo milkar 1 (100%) ban jati hain. Iske liye **Softmax Function** use hota hai. Ye mathematical roop se zyada accurate aur fast hai.

#### 4. Regularization Options
Model ko overfit hone se bachane ke liye aap parameter pass kar sakte ho `penalty='l1'`, `'l2'`, ya `'elasticnet'`.
* **L2 (Default):** Weights ko chota karta hai (Ridge Regression jaisa).
* **L1:** Jo features kaam ke nahi hain (noise), unke weights ko directly $0$ kar deta hai. Ye built-in Feature Selection ka kaam karta hai.
* **Elastic-Net:** L1 aur L2 dono ka combination. Ye tab kaam aata hai jab data me features rows (samples) se zyada hon ya features aapas me highly correlated hon.

### Logistic Regression: Solvers (Optimization Algorithms)

#### 1. Ye Solvers Kya Hote Hain?
Logistic regression model jab train hota hai, toh wo ek optimization problem solve karta hai. Iska main goal hota hai best weights/coefficients ($\theta$) find karna jo error (cost function) ko sabse kam kare. In optimal weights ko dhundhne ke liye jo under-the-hood **mathematical algorithms** use hote hain, unhe hum **Solvers** kehte hain.

#### 2. Iske Piche Ka Maths / Intuition kya hai?
Logistic Regression me humara goal **Log Loss (Cross-Entropy Loss)** $J(\theta)$ ko minimize karna hota hai. 
Basic theory me hum minimum point dhundhne ke liye **Gradient Descent** ka use karte hain. Lekin Gradient Descent kabhi-kabhi slow ho sakta hai aur local minimum me fas sakta hai. Isiliye production ya `scikit-learn` libraries me jyada advanced aur fast optimization techniques (solvers) use hoti hain jo derivative aur second-order derivative (Hessian matrix) ka use karke fast global minimum tak pohochti hain.

#### 3. Konsa Solver Kyu aur Kab Use Karein?
Scikit-learn me 5 main solvers milte hain: `'newton-cg'`, `'lbfgs'`, `'liblinear'`, `'sag'`, aur `'saga'`. Sahi solver choose karna tumhare problem setup par depend karta hai:

*   **Dataset Size ke according:**
    *   **Small Datasets:** `'liblinear'` use karo. Ye chhote data ke liye bahut efficient hai.
    *   **Large Datasets:** `'sag'` (Stochastic Average Gradient descent) aur `'saga'` use karo. Ye memory efficient hote hain aur bade data par fast convergence dete hain.

*   **Data Scaling ke according:**
    *   Agar tumhara dataset **unscaled** hai (yaani features different ranges me hain), toh `'liblinear'`, `'lbfgs'`, aur `'newton-cg'` jyada robust hain.
    *   *(Note: `'sag'` aur `'saga'` lagane se pehle data ko Standardize/Scale karna bahut zaruri hota hai, warna wo converge nahi honge.)*

*   **Multiclass Problems (Classes > 2):**
    *   **Multinomial Loss (True Multiclass):** Agar multiple classes hain aur tum true multinomial probability nikalna chahte ho, toh sirf `'newton-cg'`, `'sag'`, `'saga'`, aur `'lbfgs'` hi support karte hain.
    *   **One-Versus-Rest (OvR):** `'liblinear'` direct multinomial support nahi karta. Ye multiclass ko OvR scheme (har class vs baaki sab) me tod kar solve karta hai.

#### 4. Important Takeaways 💡
*   **Default Behavior:** Scikit-learn me logistic regression ka default solver **`'lbfgs'`** hota hai. Ye most of the general cases me best performance deta hai.
*   **Code Implementation:**
    
```python
    from sklearn.linear_model import LogisticRegression

    # Model initialization with default lbfgs solver
    logit_classifier = LogisticRegression(solver='lbfgs')
    
    # Example for large dataset
    # model_large = LogisticRegression(solver='saga')
    ```

### Logistic Regression: Regularization (Penalty)

#### 1. Regularization Kya Hai aur Kyu Use Karte Hain?
Jab humara model training data ko bahut zyada deeply yaad kar leta hai (Overfitting), toh wo naye data par achha perform nahi karta. Isse bachne ke liye hum model ko ek **"Penalty"** dete hain taaki wo apne weights ($\theta$) ko unnecessarily bada na kare. Scikit-learn me Logistic Regression by default regularization apply karta hai kyuki isse **numerical stability** bhi improve hoti hai.

#### 2. Iske Piche Ka Maths / Intuition kya hai?
Normal logistic regression me humara goal sirf Loss (Error) minimize karna hota hai. Regularization me hum Cost function me ek extra penalty term jod dete hain:
**Total Cost = Log Loss + Penalty**

Penalty main 3 tarah ki hoti hai:
*   **L1 Penalty (Lasso):** Ye weights ki **absolute value** add karta hai. $\text{Penalty} = \lambda \sum |\theta_i|$
    * *Fayda:* Ye useless features ke weights ko exactly **0** kar deta hai. Isliye ye feature selection me bahut kaam aata hai.
*   **L2 Penalty (Ridge):** Ye weights ka **square** add karta hai. $\text{Penalty} = \lambda \sum \theta_i^2$
    * *Fayda:* Ye weights ko chhota zaroor karta hai par completely 0 nahi karta. Ye model ko stable banata hai.
*   **Elastic-Net:** Ye L1 aur L2 dono ka combination use karta hai.

#### 3. Scikit-learn me Penalty Options
`penalty` parameter ko 4 values di ja sakti hain:
*   `'l2'` - L2 penalty add karta hai (Ye **default** hai).
*   `'l1'` - L1 penalty add karta hai.
*   `'elasticnet'` - L1 aur L2 dono ka mix add karta hai.
*   `'none'` - Koi regularization use nahi hogi (rarely recommended).

#### 4. Solver aur Penalty Compatibility (Kaun kiske sath kaam karta hai?)
Bhai ye sabse zyada error dene wali jagah hai kyuki har solver har penalty ko support nahi karta. 
*Rule of thumb:* **L2 penalty sabhi solvers support karte hain**, lekin **L1 aur elasticnet sirf specific solvers** handle kar sakte hain.

| Solver | Supported Penalties |
| :--- | :--- |
| **`'newton-cg'`** | `['l2', 'none']` |
| **`'lbfgs'`** (Default solver) | `['l2', 'none']` |
| **`'sag'`** | `['l2', 'none']` |
| **`'liblinear'`** | `['l1', 'l2']` |
| **`'saga'`** (Most Versatile) | `['elasticnet', 'l1', 'l2', 'none']` |

#### 5. Important Takeaways & Code Implementation 💡
*   By default, Logistic Regression **L2 penalty** use karta hai.
*   Agar tumhe `l1` ya `elasticnet` chahiye, toh tumhe solver change karke `liblinear` ya `saga` karna padega, warna scikit-learn error dega.

```python
from sklearn.linear_model import LogisticRegression

# 1. Default Implementation (uses L2 penalty and lbfgs solver)
logit_classifier = LogisticRegression(penalty='l2')

# 2. Implementation with L1 penalty (Requires compatible solver)
logit_l1 = LogisticRegression(penalty='l1', solver='liblinear')

# 3. Implementation with Elasticnet (Requires saga solver)
# (elasticnet use karne par l1_ratio parameter bhi pass karna padta hai)
logit_en = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5)
```

### Logistic Regression: Controlling Regularization (Parameter 'C')

#### 1. Ye 'C' Parameter Kya Hai?
Abhi tak humne dekha ki model me L1 ya L2 penalty lagani hai. Lekin **kitni** penalty lagani hai? Penalty ki force/strength ko control karne ke liye scikit-learn me `C` parameter ka use kiya jata hai. Ye bilkul ek volume knob ki tarah kaam karta hai.

#### 2. Iske Piche Ka Maths / Intuition kya hai?
Agar tum ML ki standard books padhoge, toh wahan regularization strength ko $\lambda$ (lambda) se dikhaya jata hai. But `scikit-learn` isme thoda alag chalta hai. 

Scikit-learn me `C` parameter use hota hai, jo ki regularization rate ($\lambda$) ka **inverse (ulta)** hota hai.
$C = \frac{1}{\lambda}$

Scikit-learn internally jo optimization problem solve karta hai wo kuch aisi dikhti hai:
$$ \arg \min_{w,c} \Big( \text{Regularization Penalty} + C \times \text{Cross-Entropy Loss} \Big) $$

Dhyan do: Yahan $C$ penalty ke sath nahi, balki Loss (Error) ke sath multiply ho raha hai. Isiliye $C$ ka behaviour standard $\lambda$ se opposite hota hai.

#### 3. Chhota 'C' vs Bada 'C' (Golden Rule)
Kyunki $C$ inverse hai, isliye iska effect yaad rakhna bahut zaroori hai:
*   **Smaller `C` (e.g., 0.001, 0.01, 0.1):** $\rightarrow$ **Stronger Regularization.** 
    * *Kyu?* Agar $C$ chhota hoga, toh equation me Loss ka weight kam ho jayega aur Penalty ka weight automatically haavi (dominant) ho jayega. 
    * *Kab use karein?* Jab tumhara model Overfit kar raha ho (training accuracy bahut high, par testing accuracy low).
*   **Larger `C` (e.g., 10, 100, 1000):** $\rightarrow$ **Weaker Regularization.**
    * *Kyu?* Agar $C$ bahut bada hoga, toh Model ka poora focus sirf aur sirf Loss ko minimize karne par chala jayega, aur Penalty ignore ho jayegi.
    * *Kab use karein?* Jab model Underfit kar raha ho aur tum chahte ho ki wo training data ko thoda aur deeply learn kare.

#### 4. Important Takeaways 💡
*   `C` ki value **hamesha positive** number honi chahiye (negative `C` error dega).
*   Scikit-learn me `C` ki default value **`1.0`** hoti hai.
*   Hyperparameter Tuning (jaise GridSearchCV) me hum $C$ ki list (like `[0.01, 0.1, 1, 10, 100]`) try karte hain best model find karne ke liye.

#### 5. Code Implementation
```python
from sklearn.linear_model import LogisticRegression

# 1. Default C=1.0 (Normal Regularization)
logit_default = LogisticRegression(C=1.0)

# 2. Strong Regularization (C is very small) - Use to fix Overfitting
logit_strong = LogisticRegression(C=0.01)

# 3. Weak Regularization (C is very large) - Model trusts the data completely
logit_weak = LogisticRegression(C=100.0)

### Logistic Regression: Handling Class Imbalance (`class_weight`)

#### 1. Ye 'class_weight' Kya Hai aur Kyu Use Karte Hain?
Maan lo tum ek Fraud Detection model bana rahe ho jisme 99% data "Normal" (Class 0) transactions ka hai, aur sirf 1% "Fraud" (Class 1) hai. Aise dataset ko **Imbalanced Dataset** kehte hain. 
Agar tum normal Logistic Regression chalaoge, toh model aalsi (lazy) ban jayega aur har transaction ko "Normal" predict karega. Fir bhi uski accuracy 99% aayegi, par model totally useless hoga! 
Is problem ko solve karne ke liye hum `class_weight` parameter ka use karte hain, taaki hum minority class (jaise Fraud) ko zyada importance/bhaav de sakein.

#### 2. Iske Piche Ka Maths / Intuition kya hai?
Normal training me, model har example par galti karne ki penalty equal maanta hai. Lekin `class_weight` ke aane se Cost/Loss function update ho jata hai:
$$ \text{Weighted Loss} = w_c \times \text{Loss} $$
Jahan $w_c$ us particular class ka weight hai. 
Agar ek class par humne **higher value (weight)** set kiya hai, toh us class par mistake (misclassification) karne par model ko **bahut heavy penalty** padegi. Penalty kam karne ke chakkr me optimization solver us class par **higher emphasis (zyada dhyaan)** dega. 

#### 3. Ise Use Kaise Karein? (Options)
`class_weight` parameter me 3 tarah ki values di ja sakti hain:
*   **`None` (Default):** Sabhi classes ka weight 1 hota hai. Model sabko equally treat karta hai.
*   **`'balanced'` (Highly Recommended):** Scikit-learn automatically data frequency ko analyze karta hai aur **inverse weights** assign karta hai. Jo class kam hogi, uska weight automatically bada ho jayega.
    * *Math behind 'balanced':* $w_j = \frac{\text{Total Samples}}{\text{Number of Classes} \times \text{Samples in Class } j}$
*   **Custom Dictionary:** Tum khud directly weights define kar sakte ho. E.g., `{0: 1, 1: 10}` (Class 0 ka weight 1, aur Class 1 ka weight 10).

#### 4. Important Takeaways 💡
*   Ye parameter sirf Logistic Regression ka nahi, balki `sklearn` ke lagbhag **sabhi classifiers** (jaise Decision Trees, Random Forest, SVM) me available hota hai.
*   *(Stack Overflow tip)*: Jab bhi tum `class_weight` use karo imbalanced data pe, toh model ki performance measure karne ke liye kabhi bhi `Accuracy` ka use mat karna. Hamesha `Precision`, `Recall`, aur `F1-Score` (Classification Report) dekhna.

#### 5. Code Implementation
```python
from sklearn.linear_model import LogisticRegression

# 1. Standard Logistic Regression (Assumes perfectly balanced data)
logit_default = LogisticRegression(class_weight=None)

# 2. Auto-balanced (Best for imbalanced data like Fraud, Cancer detection)
logit_balanced = LogisticRegression(class_weight='balanced')

# 3. Custom Weights (Class 1 is 5 times more important than Class 0)
custom_weights = {0: 1.0, 1: 5.0}
logit_custom = LogisticRegression(class_weight=custom_weights)

### Advanced Implementations: LogisticRegressionCV & SGDClassifier

#### 1. LogisticRegressionCV (Auto-Tuning the Model)
**Ye kya hai aur kyu use karein?**
Normally hum best `C` (regularization strength) aur `l1_ratio` dhundhne ke liye `GridSearchCV` ka use karte hain jo ki time-consuming hota hai. Lekin `LogisticRegressionCV` iska ek highly optimized, in-built version hai. Tumhe bas isko batana hota hai ki kitne folds (CV) banane hain, aur ye under-the-hood bahut efficiently best parameters dhoondh nikalta hai.

**Iske Piche Ka Intuition:**
Ye automatically data ko $k$ hisso (folds) me split karta hai. Har fold par ye internally alag-alag `C` aur `l1_ratio` ki grid test karta hai. Jo value tumhare specified `scoring` metric (jaise 'accuracy', 'f1', ya 'roc_auc') pe sabse best average score deti hai, model automatically us best value ke sath final training kar leta hai.

#### 2. SGDClassifier (Logistic Regression for Massive Data)
**Ye kya hai aur kyu use karein?**
Scikit-learn me `SGDClassifier` ek generic machine learning algorithm API hai. Agar tum iske constructor me `loss='log_loss'` set kar do, toh ye exact wahi mathematical model ban jata hai jo Logistic Regression hai.
*Toh fir ise alag se kyu banaya?* Standard Logistic Regression solvers (jaise `lbfgs` ya `newton-cg`) poore dataset ko ek sath RAM me load karke matrix operations karte hain. Agar tumhare paas 10 Million rows (bahut bada data) hain, toh tumhara system crash ho jayega (Out of Memory). `SGDClassifier` data ko ek-ek row (ya chote batches) me process karta hai.

**Iske Piche Ka Maths (Stochastic Gradient Descent):**
Standard Gradient descent poore data ka error calculate karke weights update karta hai. SGD sirf ek random data point $(x^{(i)}, y^{(i)})$ ka use karke Loss $J(\theta)$ ka derivative nikalta hai aur weights ko turant update kar deta hai:
$$ \theta_j := \theta_j - \alpha \frac{\partial}{\partial \theta_j} J(\theta; x^{(i)}, y^{(i)}) $$
*(Yahan $\alpha$ learning rate hai. Ye path thoda zig-zag hota hai par massive datasets pe infinitely faster hota hai).*

#### 3. Important Takeaways 💡 (Kab kya use karein?)
*   **Normal Dataset (Fit in RAM) + Best Hyperparameters chahiye:** `LogisticRegressionCV` use karo. Ye GridSearch se zyada efficient hai.
*   **Huge Dataset (Out-of-core learning / Big Data):** `SGDClassifier(loss='log_loss')` use karo. Ise tum `partial_fit()` method ke sath chunks me train kar sakte ho.

#### 4. Code Implementation
```python
from sklearn.linear_model import LogisticRegressionCV, SGDClassifier

# --- 1. LogisticRegressionCV Example ---
# Cs=10 means it will test 10 logarithmically spaced values of C
# cv=5 means 5-fold cross-validation
# n_jobs=-1 uses all CPU cores for faster computation
logit_cv = LogisticRegressionCV(Cs=10, cv=5, scoring='accuracy', n_jobs=-1)
# logit_cv.fit(X_train, y_train) 
# Training ke baad logit_cv.C_ type karke tum selected best C dekh sakte ho.

# --- 2. SGDClassifier Example ---
# Setting loss='log_loss' converts this generic SGD into a Logistic Regression model.
# (Note: In older sklearn versions < 1.1, the parameter was loss='log')
sgd_logit = SGDClassifier(loss='log_loss', penalty='l2', max_iter=1000)
# sgd_logit.fit(X_train, y_train)

### SGDClassifier: The Ultimate Masterclass (Deep Math & Intuition)

#### 1. SGDClassifier Asal Me Hai Kya?
Beginners hamesha sochte hain ki SGD ek naya Algorithm hai. **Nahi!** 
Stochastic Gradient Descent (SGD) sirf ek **Optimization Technique** hai. Scikit-learn me `SGDClassifier` ek "Khali Dhancha" (Blank Canvas) ki tarah hai. Tum iske andar jo `loss` function daloge, ye wahi machine learning model ban jayega. Ye ek chameleon (girgit) ki tarah hai jo loss parameter ke hisaab se apna roop badal leta hai.

#### 2. The Deep Math: Convex Loss Functions & SGD
Lecture me ek word hai: **Convex Loss Functions**. Ye optimization ka sabse important word hai.
*   **Convex ka matlab:** Ek aisa curve jo ek "Bowl" (katori) ki tarah dikhta ho. Iski sabse badi mathematical property ye hai ki isme koi **Local Minima nahi hota**, sirf ek **Global Minimum** hota hai. Yani SGD optimization kabhi kisi galat gaddhe (trap) me nahi fasega aur hamesha best weights ($\theta$) dhoondh nikalega.

*   **SGD vs Batch Gradient Descent (The Math):**
    *   *Normal (Batch) GD:* Har step me loss ka derivative poore dataset (size $N$) par nikalta hai. 
        $$ \theta_j := \theta_j - \alpha \frac{1}{N} \sum_{i=1}^{N} \frac{\partial}{\partial \theta_j} L(y^{(i)}, \hat{y}^{(i)}) $$
    *   *Stochastic GD (SGD):* Har step me derivative sirf **ek random row** (single example) par nikalta hai. Ye path thoda noisy/zig-zag hota hai par infinitely faster hota hai.
        $$ \theta_j := \theta_j - \alpha \frac{\partial}{\partial \theta_j} L(y^{(i)}, \hat{y}^{(i)}) $$

#### 3. Handling Scalability & Sparse Data ($>10^5$ examples)
*   **Massive Data:** Kyunki ye ek baar me sirf ek data point (ya mini-batch) dekhta hai, isko poora 100GB ka data RAM me load karne ki zarurat nahi hai. Ye `partial_fit()` use karke infinite data par train ho sakta hai.
*   **Sparse Machine Learning (NLP/Text):** Text classification me jab hum TF-IDF lagate hain, toh aisi matrix banti hai jisme 99% values zero (0) hoti hain (Sparse Matrix). SGD itna smart hai ki agar kisi feature ki value 0 hai $x_j = 0$, toh ye uske gradient ka calculation directly skip kar deta hai. Isse Text Classification (NLP) me computation 100x fast ho jati hai.

#### 4. Multi-class Classification: One-Versus-All (OVA)
Under the hood, `SGDClassifier` natively sirf Binary Classification (2 classes) karta hai. 
Agar tumhare paas 10 classes hain (0 to 9), toh ye internally **OVA (One-vs-All)** scheme lagata hai.
*   *Math Logic:* Ye 10 alag-alag binary classifiers banayega. 
    * Classifier 1: "Is this Class 0 or NOT Class 0?"
    * Classifier 2: "Is this Class 1 or NOT Class 1?"
*   *Prediction:* Har classifier apna probability/score dega $f_k(x)$. Final answer wo class hogi jiska formula maximize hoga: 
    $$ \hat{y} = \arg\max_{k \in \{1...K\}} f_k(x) $$

#### 5. The Heart of SGD: The `loss` Parameter (Full Math)
Yahi wo parameter hai jo tay karta hai ki `SGDClassifier` kya banega.
Maan lo humara model ek raw score (margin) predict karta hai: $z = w^T x + b$
Aur true label hai $y \in \{-1, 1\}$. $y \cdot z$ ko hum **Margin** kehte hain. Agar $y \cdot z > 0$, yani tumhari prediction sahi direction me hai.

*   **`loss = 'hinge'` (Linear Support Vector Machine - SVM)**
    *   *Math:* $$ L(y, z) = \max(0, 1 - y \cdot z) $$
    *   *Deep Intuition:* Ye Soft-Margin SVM banata hai. Agar model ki prediction ekdam confident hai ($y \cdot z \ge 1$), toh loss exactly 0 ho jata hai. Agar prediction galat hai ya confidence kam hai (< 1), toh penalty linearly badhti hai. Ye sirf boundary wale points (Support Vectors) pe focus karta hai.
    
*   **`loss = 'log_loss'` (Logistic Regression)**
    *   *Math:* $$ L(y, z) = \log(1 + e^{-y \cdot z}) $$
    *   *Deep Intuition:* Ye Hinge ki tarah zero nahi hota. Ye har point ko thoda bahut penalty deta hai, chahe wo correct kyu na ho. Is wajah se, SVM ke comparison me, ye ekdam accurate **Probability Estimates** (predict_proba) nikal pata hai.

*   **`loss = 'modified_huber'` (Smoothed Hinge Loss)**
    *   *Deep Intuition:* Hinge loss margin=1 par ek sharp corner banata hai jahan math me derivative (gradient) fail ho jata hai. Modified Huber us sharp corner ko ek quadratic (U-shape) curve se "smooth" kar deta hai. 
    *   *Fayda:* Ye Outliers (kharaab data points) ke liye bahut tolerant hota hai aur 'hinge' ke comparison me probability estimates bhi nikal pata hai.

*   **`loss = 'squared_hinge'`**
    *   *Math:* $$ L(y, z) = (\max(0, 1 - y \cdot z))^2 $$
    *   *Deep Intuition:* Ye SVM (hinge) hi hai, bas galti karne par punishment quadratic (square me) milti hai. Ye model ko force karta hai ki wo galat predictions (margin < 1) ko bahut jaldi theek kare.

*   **`loss = 'perceptron'`**
    *   *Math:* $$ L(y, z) = \max(0, -y \cdot z) $$
    *   *Deep Intuition:* Ye classic neural network ka perceptron algorithm banata hai. Isme "margin" ka koi concept nahi hai. Agar prediction slightly bhi sahi direction me hai ($y \cdot z > 0$), toh loss seedha 0. Ye converge hone me SVM se fast hai par accuracy thodi kam ho sakti hai.

*   **Regression Losses:**
    *   Agar tumhe Classification ki jagah Regression (continuous values predict) karni hai (like predicting house prices), toh tum `SGDRegressor` import karke usme `'squared_error'`, `'huber'`, ya `'epsilon_insensitive'` loss laga sakte ho.

#### 6. Ultimate Code Implementation
```python
from sklearn.linear_model import SGDClassifier

# 1. Building a Linear SVM (Excellent for text classification)
svm_model = SGDClassifier(loss='hinge', penalty='l2', max_iter=1000)

# 2. Building a Logistic Regression (When you need predict_proba probabilities)
logit_model = SGDClassifier(loss='log_loss', penalty='l2', max_iter=1000)

# 3. Handling Outliers in Data
robust_model = SGDClassifier(loss='modified_huber', penalty='l2')

# Training (Batch vs SGD)
# Standard fit (uses batch/epochs internally)
svm_model.fit(X_train, y_train)

# For massive >10^5 data that doesn't fit in RAM
# svm_model.partial_fit(X_chunk, y_chunk, classes=[0, 1, 2])